# ShopAssist: NLP-Based E-commerce Intent Classification

ShopAssist is an e-commerce chatbot prototype that uses a feedforward neural network (FNN) to classify customer messages into 19 predefined intents.

## Project Overview

The project uses TF-IDF to convert text into numerical features and a trained neural network to predict the customer's intent. Based on the predicted intent, a response router provides an appropriate response or performs a basic operation using mock order data.

## Key Components

- Dataset preparation and intent selection
- TF-IDF text feature extraction
- Feedforward neural network for intent classification
- Model evaluation and prediction confidence threshold
- Model and preprocessing artifact persistence
- Intent-based response routing using mock data

In [36]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input
from tensorflow.keras.activations import relu,linear
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
import numpy as np
import re

## 1. Dataset Loading and Exploration

The project uses the Bitext retail and e-commerce chatbot training dataset.

The dataset contains customer utterances labelled with their corresponding intents. The original dataset is preserved in `original_df` for exploration, while a filtered DataFrame is used for model development.

In [3]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset")

d:\Projects\shop_assist\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [47]:
original_df=pd.DataFrame(ds["train"])
print(original_df.head())
original_df.info()
print("Dataset shape:", original_df.shape)
print(original_df[(original_df["intent"]=="customer_service")|(original_df["intent"]=="human_agent")])

                                         instruction       intent category  \
0                   I got to add an item to the cart  add_product     CART   
1  wanna add fucking products to the basket can h...  add_product     CART   
2    i have to add products to the basket i ned help  add_product     CART   
3  di like to add products to the cart could i ge...  add_product     CART   
4  I need to add an item to the cart , where do I...  add_product     CART   

       tags                                           response  
0        BL  I'll get right on it! I'm here to assist you i...  
1   BCIMQWZ  I sincerely apologize if you've encountered an...  
2     BCMQZ  You bet! I'm here to assist you in adding prod...  
3  BCILMPQZ  Indeed! I'm here to assist you in adding produ...  
4     BCILZ  I'll take care of it! I'm here to help you wit...  
<class 'pandas.DataFrame'>
RangeIndex: 44884 entries, 0 to 44883
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
--- 

In [5]:
instruction_intent=pd.DataFrame(original_df[["instruction","intent"]])
print(instruction_intent["intent"].value_counts())

intent
request_invoice                   1000
refund_status                      999
use_app                            999
cancel_order                       996
delivery_issue                     996
payment_issue                      996
track_delivery                     996
wrong_item                         996
close_account                      995
payment_methods                    995
sales_period                       995
submit_product_idea                995
human_agent                        994
refund_policy                      994
return_policy                      994
return_product                     994
return_product_in_store            994
submit_product_feedback            994
technical_issue                    994
availability_online                993
recover_password                   993
remove_product                     993
return_product_online              993
customer_service                   992
damaged_delivery                   992
product_issue     

## 2. Intent Selection and Dataset Filtering

ShopAssist focuses on 19 predefined intents related to e-commerce customer support.

The original dataset is filtered to retain only messages belonging to these intents. This creates `selected_df`, which is used for training and evaluating the intent classification model.

The number of samples and the distribution of intents are examined to understand the size and balance of the selected dataset.

In [6]:
selected_intents = [
    "track_order",
    "cancel_order",
    "order_history",
    "delivery_issue",
    "delivery_time",
    "missing_item",
    "damaged_delivery",
    "refund_status",
    "refund_policy",
    "return_policy",
    "return_product",
    "request_refund",
    "add_product",
    "remove_product",
    "availability",
    "product_information",
    "payment_issue",
    "payment_methods",
    "human_agent"
]
selected_df=original_df[original_df["intent"].isin(selected_intents)]
print(selected_df.shape)
print(selected_df["intent"].value_counts())

(18433, 5)
intent
refund_status          999
cancel_order           996
delivery_issue         996
payment_issue          996
payment_methods        995
human_agent            994
refund_policy          994
return_policy          994
return_product         994
remove_product         993
damaged_delivery       992
order_history          988
track_order            988
product_information    987
availability           972
add_product            957
request_refund         957
delivery_time          920
missing_item           721
Name: count, dtype: int64


In [38]:
print("Duplicate messages:")
print(selected_df["instruction"].duplicated().sum())

print("\nMissing values:")
print(selected_df.isnull().sum())

print("\nExamples of potentially overlapping intents:")
print(
    selected_df[
        selected_df["intent"].isin([
            "delivery_issue",
            "missing_item",
            "damaged_delivery"
        ])
    ][["intent", "instruction"]].head(15)
)

Duplicate messages:
0

Missing values:
instruction    0
intent         0
category       0
tags           0
response       0
dtype: int64

Examples of potentially overlapping intents:
                intent                                        instruction
8609  damaged_delivery  one of the items I bought wzs damaged in shipp...
8610  damaged_delivery  could ya help me reporting an fucking damaged ...
8611  damaged_delivery  the fucking package i received was damaged cou...
8612  damaged_delivery             how do I report ufcking crushed items?
8613  damaged_delivery  one of the items i got was fucking damaged can...
8614  damaged_delivery  one of the items is fucking damaged could ya r...
8615  damaged_delivery  i have to report an fucking crushed item how c...
8616  damaged_delivery  the package I purchased was damaegd, I would l...
8617  damaged_delivery  some products I received was damaged, how can ...
8618  damaged_delivery  the fucking packae I ordered was crushed, how ...
861

## 3. Training, Validation, and Test Sets

The selected dataset is divided into three subsets:

- **Training set (80%):** Used to train the neural network.
- **Validation set (10%):** Used to monitor model performance during training.
- **Test set (10%):** Used for the final evaluation of the trained model.

Stratified splitting is used to preserve the approximate distribution of the 19 intents across all three subsets.

A fixed random state is used to make the split reproducible.

In [40]:
X_train, X_, y_train, y_ = train_test_split(
    selected_df["instruction"],
    selected_df["intent"],
    test_size=0.2,
    random_state=1,
    stratify=selected_df["intent"]
)

X_cv, X_test, y_cv, y_test = train_test_split(
    X_,
    y_,
    test_size=0.5,
    random_state=1,
    stratify=y_
)

print("Training set:", X_train.shape)
print("Validation set:", X_cv.shape)
print("Test set:", X_test.shape)

Training set: (14746,)
Validation set: (1843,)
Test set: (1844,)


## 4. Text Preprocessing Using TF-IDF

TF-IDF (Term Frequency–Inverse Document Frequency) converts customer messages into numerical feature vectors that can be used as input to the neural network.

The vectorizer is fitted on the training messages to learn the vocabulary and IDF weights. The same fitted vectorizer is then used to transform the validation and test messages.

This ensures that all three datasets use the same feature representation and prevents information from the validation and test sets from influencing vocabulary fitting.

In [ ]:
vectorizer=TfidfVectorizer()
text_to_num_train=vectorizer.fit_transform(X_train)
text_to_num_cv=vectorizer.transform(X_cv)
text_to_num_test=vectorizer.transform(X_test)
print("Training features:", text_to_num_train.shape)
print("Validation features:", text_to_num_cv.shape)
print("Test features:", text_to_num_test.shape)

Training features: (14746, 2597)
Validation features: (1843, 2597)
Test features: (1844, 2597)


## 5. Intent Label Encoding

The neural network requires numerical labels instead of text-based intent names.

Each of the 19 intents is assigned a unique integer index. The same mapping is applied to the training, validation, and test labels.

The order of the intent list is preserved so that each model output index can later be mapped back to its corresponding intent name.

In [45]:
intent_to_index={}
count=0
for intents in selected_intents:
    intent_to_index[intents]=count
    count=count+1
y_train=y_train.map(intent_to_index)
y_test=y_test.map(intent_to_index)
y_cv=y_cv.map(intent_to_index)

In [10]:
print(y_train[0:5])
print(y_test[0:5])
print(y_cv[0:5])

27437    11
29490     9
20826    15
30010     9
24514     7
Name: intent, dtype: int64
42150     0
11472     4
14428    18
14639     5
26111    13
Name: intent, dtype: int64
9069      6
30551    10
30370    10
14059    18
4263      1
Name: intent, dtype: int64


In [43]:
print("Training set:", y_train.shape)
print("Validation set:", y_cv.shape)
print("Test set:", y_test.shape)

Training set: (14746,)
Validation set: (1843,)
Test set: (1844,)


## 6. Feedforward Neural Network Architecture

A feedforward neural network (FNN) is used to classify customer messages into 19 predefined intents.

### Architecture

- **Input:** 2,597 TF-IDF features per message.
- **Hidden layer 1:** 128 neurons with ReLU activation.
- **Hidden layer 2:** 128 neurons with ReLU activation.
- **Hidden layer 3:** 64 neurons with ReLU activation.
- **Output layer:** 19 neurons with linear activation, producing one logit for each intent.

The model uses the numerical TF-IDF representation of each message as input. During inference, softmax converts the output logits into probabilities over the 19 intent classes.

In [15]:
model=Sequential([
    Input(shape=(text_to_num_train.shape[1],)),
    Dense(units=128,activation="relu"),
    Dense(units=128,activation="relu"),
    Dense(units=64,activation="relu"),
    Dense(units=19,activation="linear")
])

## 7. Model Training

The feedforward neural network is trained using the TF-IDF training features and their corresponding integer intent labels.

Sparse categorical cross-entropy is used as the loss function because the target labels are integer class indices. The model produces logits, so the loss is configured with `from_logits=True`.

Validation data is supplied during training to monitor the model's performance on unseen examples.

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [ ]:
model.fit(
    text_to_num_train,
    y_train,
    epochs=25,
    validation_data=(text_to_num_cv, y_cv)
)

Epoch 1/25
461/461 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9525 - loss: 0.1635 - val_accuracy: 0.9924 - val_loss: 0.0234
Epoch 2/25
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9961 - loss: 0.0180 - val_accuracy: 0.9946 - val_loss: 0.0293
Epoch 3/25
461/461 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9936 - loss: 0.0255 - val_accuracy: 0.9902 - val_loss: 0.0541
Epoch 4/25
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9944 - loss: 0.0436 - val_accuracy: 0.9929 - val_loss: 0.0465
Epoch 5/25
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9957 - loss: 0.0294 - val_accuracy: 0.9935 - val_loss: 0.0457
Epoch 6/25
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9996 - loss: 0.0044 - val_accuracy: 0.9946 - val_loss: 0.0383
Epoch 7/25
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9999 - loss: 0.0020 - val_accuracy: 0.9946 - val_loss: 0.0398
Epoch 8/25
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.0000 - loss: 5.1424e-06 - val_accuracy

## 8. Model Evaluation

The trained model is evaluated on the validation and test sets.

- **Validation evaluation:** Measures performance on the validation set.
- **Test evaluation:** Measures final performance on held-out examples that were not used to train the model.

Accuracy measures the proportion of correctly classified messages, while loss measures prediction error using the configured loss function.

The test set is reserved for the final evaluation rather than model tuning.

In [20]:
model.evaluate(text_to_num_cv, y_cv)

58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9957 - loss: 0.0393  


[0.039319854229688644, 0.9956592321395874]

In [21]:
model.evaluate(text_to_num_test, y_test)

58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9930 - loss: 0.0385


[0.03851485624909401, 0.9929500818252563]

## 9. Saving and Reloading Model Artifacts

The trained model and its preprocessing components are saved so that they can be reused without retraining.

The project saves three artifacts:

- `shopassist_model.keras`: The trained neural network, including its architecture and weights.
- `tfidf_vectorizer.pkl`: The fitted TF-IDF vectorizer, including its learned vocabulary and IDF values.
- `selected_intent.json`: The ordered list of intent labels used to map model output indices back to intent names.

During inference, the saved model, vectorizer, and intent list are loaded to ensure that prediction uses the same components as the trained system.

In [21]:
model.save("shopassist_model.keras")

In [22]:
import joblib

joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [25]:
import json

with open("selected_intent.json", "w") as f:
    json.dump(selected_intents, f)

In [ ]:
from tensorflow import keras
import joblib
import json

model = keras.models.load_model("shopassist_model.keras")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

with open("selected_intent.json", "r") as f:
    selected_intents = json.load(f)

print("Model loaded:", model is not None)
print("Vectorizer loaded:", vectorizer is not None)
print("Number of intents:", len(selected_intents))

Model loaded: True
Vectorizer loaded: True
Number of intents: 19


## 10. Mock Data and Intent-Based Response Routing

The response router connects the predicted intent to an appropriate response or basic operation.

The prototype supports:

- **Informational responses:** Return policy, refund policy, and payment methods.
- **Order lookups:** Tracking, refund status, and delivery-related queries.
- **Basic operations:** Cancellation and return/refund requests.
- **Product-related responses:** Sample product information and availability.
- **Issue handling:** Basic escalation messages for delivery, missing-item, damage, and payment issues.

Order details are stored in an in-memory Python dictionary. This mock data demonstrates how an intent classifier can be integrated into an application without requiring a production database.

In [14]:
mock_orders = {
    "AB1234": {
        "product": "Wireless Headphones",
        "status": "In Transit",
        "estimated_delivery": "18 October 2026",
        "refund_status": "Not Requested"
    },
    "CD5678": {
        "product": "Running Shoes",
        "status": "Delivered",
        "estimated_delivery": "10 October 2026",
        "refund_status": "Refund Completed"
    },
    "EF9012": {
        "product": "Laptop Stand",
        "status": "Processing",
        "estimated_delivery": "20 October 2026",
        "refund_status": "Not Requested"
    },
    "GH3456": {
        "product": "Mechanical Keyboard",
        "status": "Shipped",
        "estimated_delivery": "17 October 2026",
        "refund_status": "Refund Pending"
    },
    "JK7890": {
        "product": "USB-C Hub",
        "status": "Out for Delivery",
        "estimated_delivery": "14 October 2026",
        "refund_status": "Not Requested"
    }
}


def extraction(order_id):
    match = re.search(r"[A-Z]{2}\d{4}", order_id)

    if match is not None:
        return match.group(0)

    order_id = input("Enter order id of the format XX1234").strip()

    def check_length(id):
        if len(id) == 6:
            return True
        return False

    def check_expression(id):
        if re.fullmatch(r"[A-Z]{2}\d{4}", id) is None:
            return False
        else:
            return True

    while not (check_length(order_id) and check_expression(order_id)):
        if check_length(order_id) is False:
            order_id = input(
                "Enter order id of appropriate length"
            ).strip()
        elif check_expression(order_id) is False:
            order_id = input(
                "Enter order id of appropriate format i.e XX1234"
            ).strip()

    return order_id


def response_router(intent, p1):
    match intent:

        case "refund_policy":
            print(
                "Refund amount is generally reimbursed within "
                "4-5 business days into your respective bank account."
            )
            return

        case "return_policy":
            print(
                "Any item can be returned within 7 business days "
                "from the day of delivery"
            )
            return

        case "payment_methods":
            print("We accept debit/credit cards, UPI and cash on delivery")
            return

        case "track_order":
            order_id = extraction(p1)

            if mock_orders.get(order_id) is None:
                print("The order id entered is not present with us kindly check")
            else:
                values = mock_orders.get(order_id)
                print(
                    "The delivery status is " + values.get("status")
                    + " and will be delivered by "
                    + values.get("estimated_delivery")
                )
            return

        case "order_history":
            print("Here are the sample orders available in our system")

            for order_id, values in mock_orders.items():
                print(
                    order_id + " - " + values.get("product")
                    + " - " + values.get("status")
                    + " - " + values.get("estimated_delivery")
                )
                print("\n")
            return

        case "refund_status":
            order_id = extraction(p1)

            if mock_orders.get(order_id) is None:
                print("The order id entered is not present with us kindly check")
            else:
                values = mock_orders.get(order_id)
                print(
                    "The refund status is " + values.get("refund_status")
                    + " and the estimated delivery date is "
                    + values.get("estimated_delivery")
                )
            return

        case "availability":
            print("Here are the sample products available in our system")

            for order_id, values in mock_orders.items():
                print(order_id + " - " + values.get("product"))
                print("\n")
            return

        case "cancel_order":
            order_id = extraction(p1)

            if mock_orders.get(order_id) is None:
                print("The order id entered is not present with us kindly check")
            else:
                mock_orders.pop(order_id)
                print("The order with the " + order_id + " has been cancelled")
            return

        case "return_product":
            order_id = extraction(p1)

            if mock_orders.get(order_id) is None:
                print("The order id entered is not present with us kindly check")
            else:
                values = mock_orders.get(order_id)
                values["refund_status"] = "Refund Pending"
                print(
                    "The return for order " + order_id
                    + " has been initiated with the refund process"
                )
            return

        case "request_refund":
            order_id = extraction(p1)

            if mock_orders.get(order_id) is None:
                print("The order id entered is not present with us kindly check")
            else:
                values = mock_orders.get(order_id)
                values["refund_status"] = "Refund Pending"
                print(
                    "The order with the " + order_id
                    + " has been initiated with the refund process"
                )
            return

        case "add_product":
            print("The sample cannot add products yet")
            return

        case "remove_product":
            print("The sample cannot remove products yet")
            return

        case "delivery_issue":
            order_id=extraction(p1)

            if mock_orders.get(order_id) is None:
                print("The order id entered is not present with us kindly check")
            else:
                values = mock_orders.get(order_id)
                print(
                    "The order with the " + order_id
                    + " will be delivered by " + values.get("estimated_delivery")
                )
            return

        case "missing_item":
            print("The system will check and our team will call you back escalating the issue")
            return

        case "damaged_delivery":
            print("The system will check and our team will call you back escalating the issue")
            return

        case "payment_issue":
            print("The system will check and our team will call you back escalating the issue")
            return

        case "delivery_time":
            order_id=extraction(p1)

            if mock_orders.get(order_id) is None:
                print("The order id entered is not present with us kindly check")
            else:
                values = mock_orders.get(order_id)
                print(
                    "The order with the " + order_id
                    + " will be delivered by " + values.get("estimated_delivery")
                )
            return

        case "product_information":
            order_id=extraction(p1)

            if mock_orders.get(order_id) is None:
                print("The order id entered is not present with us kindly check")
            else:
                values = mock_orders.get(order_id)
                print(
                    "The order with the " + order_id
                    + " has the follwing product " + values.get("product")
                )
            return

        case "human_agent":
            print("The system will check and our team will call you back escalating the issue")
            return

## 11. Intent Prediction and Response Generation

The inference pipeline processes a customer message using the saved model and preprocessing artifacts.

1. **Input validation:** Empty messages are rejected.
2. **TF-IDF transformation:** The message is converted into numerical features using the fitted vectorizer.
3. **Intent prediction:** The trained FNN generates logits for the 19 intent classes.
4. **Probability calculation:** Softmax converts the logits into probabilities.
5. **Confidence threshold:** Predictions with a maximum probability below 0.80 are not routed.
6. **Intent mapping:** The predicted class index is mapped to its corresponding intent name.
7. **Response generation:** The response router handles the predicted intent.

In [ ]:
def main():
    p1=input("Please enter your query")
    p1=p1.strip()
    if p1=="":
        print("Please do not send an empty message")
    else:
        p1_num=vectorizer.transform([p1])
        output_logits=model.predict(p1_num)
        output_prob=tf.nn.softmax(output_logits)
        value=np.max(output_prob)
        if value >= 0.8:
            index=np.argmax(output_prob)
            response_router(selected_intents[index],p1)
        else:
            print("Please clarify or rephrase your message")
main()